# Pokémon Battles Prediction · Team M⁴ · Final submissions

**Team M⁴:** Matteo Schirru, Alessia Manfredi, Daniela Madonna, Margherita Guerrieri (Sapienza University of Rome)

This notebook reproduces our three final submissions to the
[FDS: Pokémon Battles Prediction 2026](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026) competition.
All the code is in our GitHub repository
[guerrieri2332706/fds-pokemon-battles-M-](https://github.com/guerrieri2332706/fds-pokemon-battles-M-):
the notebook clones it, imports the functions of our module `pokemon_module.py` and only calls them.

| Output file | Model | Features | CV accuracy |
|---|---|---|---|
| `submission_lr.csv` | Logistic regression (standardised, L2) | 154 (version 6) | 67.86% |
| `submission_gb.csv` | Histogram gradient boosting | 159 (version 7) | 67.62% |
| `submission_rf.csv` | Random forest stacked on a logistic score | 296 (forest version 2) | 67.78% |

CV accuracy: mean over the same 5 stratified folds (seed 42) for every model.
The notebook runs end to end in a few minutes on Kaggle's CPU; **Internet must be on** (notebook settings) to clone the repository.

## 1. Setup: clone the repository and import our code

As in the course demo notebook: (1) clone the public repository, (2) add its folder to `sys.path`,
(3) import the functions from our module `pokemon_module.py`.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/guerrieri2332706/fds-pokemon-battles-M-.git

In [ ]:
# 2. Update the system path, so that Python finds our module inside the cloned folder
import sys

repo_path = '/kaggle/working/fds-pokemon-battles-M-'
if repo_path not in sys.path:
    sys.path.append(repo_path)

In [ ]:
# 3. Import our module (all the code of the three submissions lives in the repository)
import numpy as np
import pandas as pd
import sklearn

from pokemon_module import (
    SEED,
    # data
    load_competition_data, find_data_dir, check_data, write_submission,
    # lookup tables and features
    build_lookups, build_logistic_features, feature_columns, build_forest_features,
    # models
    logistic_plain, logistic_scaled, gradient_boosting_search,
    random_forest, random_forest_search, logistic_sparse, stacked_forest,
    # validation
    make_cv, cv_accuracy, grid_fold_scores, fold_table,
)

print(f'Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · '
      f'scikit-learn {sklearn.__version__} · seed {SEED}')

## 2. Data

The competition provides 10,000 labelled battles and 5,000 test battles. For each battle we know Player 1's
full team, Player 2's lead and the first 30 turns (the battles last 30 to 80 turns); `player_won = 1` means
that Player 1 won. `check_data` verifies ids, target and submission template.

`build_lookups` builds the only tables shared across battles (species → types and base stats, the 58 frequent
moves, the 41 frequent species) **from the training set only** and without the target, plus the Gen 2 type chart
provided by the competition. Every feature function then works on one battle at a time, so applying it to the
test set cannot leak information.

In [ ]:
data = load_competition_data(find_data_dir('/kaggle/input'))
check_data(data)

y = pd.Series([b['player_won'] for b in data.train], name='player_won')
test_ids = [b['battle_id'] for b in data.test]

lookups = build_lookups(data.train, data.type_chart)
print(f'Species with known types: {len(lookups.species_types)}   '
      f'Frequent moves: {len(lookups.frequent_moves)}   Frequent species: {len(lookups.frequent_species)}')

## 3. Validation strategy

Every model is compared on the **same 5-fold stratified cross-validation** (`StratifiedKFold`, shuffled, seed 42),
defined once in `make_cv`. The classes are balanced, so always predicting the same class scores 50%.
A change (new features or other settings) was kept only if it raised the mean accuracy and won in most folds;
the public leaderboard never guided our choices. Standardisation and stacking live inside scikit-learn estimators,
so they are refitted on the training part of each fold.

In [ ]:
cv = make_cv()
for k, (train_idx, valid_idx) in enumerate(cv.split(np.zeros(len(y)), y), start=1):
    print(f'fold {k}: {len(train_idx)} training / {len(valid_idx)} validation battles, '
          f'Player 1 wins {y.iloc[valid_idx].mean():.1%} of the validation battles')

## 4. Features of the logistic regression (versions 1–7)

Each version adds one group of features to the previous one and tests one idea about what decides a battle:

| Version | Added features |
|---|---|
| v1 | pre-battle base stats (Player 1's team means, Player 2's lead) |
| v2 | HP of the two active Pokémon at turn 30 |
| v3 | Pokémon lost by Player 1 |
| v4 | Player 2's losses, status conditions of each team, stat boosts of the active Pokémon (from here the features are standardised) |
| v5 | Pokémon in good shape (at least 50% HP, no status) |
| v6 | whole-team HP and its trend from turn 20 to 30, super-effective and resisted hits (type chart), uses of 58 frequent moves |
| v7 | immunities: attacks wasted on immune Pokémon, immune matchups at turn 30 |

The table below recomputes the cross-validated accuracy of each version.
Version 7 does not help the logistic regression, so it is used only by gradient boosting.

In [ ]:
lr_train = build_logistic_features(data.train, lookups)
lr_test = build_logistic_features(data.test, lookups)

history = []
for version, table in lr_train.items():
    cols = feature_columns(table)
    scaled = version not in ('v1', 'v2', 'v3')
    model = logistic_scaled() if scaled else logistic_plain()
    scores = cv_accuracy(model, table[cols], y, cv)
    if version == 'v6':
        scores_lr = scores
    history.append({'version': version, 'features': len(cols),
                    'model': 'standardised logistic' if scaled else 'logistic',
                    'CV accuracy': scores.mean(), 'std across folds': scores.std()})

display(pd.DataFrame(history).style.format({'CV accuracy': '{:.2%}', 'std across folds': '{:.2%}'})
        .hide(axis='index'))

### Submission 1 · Logistic regression (version 6)

`StandardScaler` + L2 `LogisticRegression` (`max_iter=1000`, seed 42) on the 154 features of version 6,
refitted on all 10,000 labelled battles. Cross-validated accuracy 67.86%.
On the leaderboard: 68.67% public, 66.97% private.

In [ ]:
cols_v6 = feature_columns(lr_train['v6'])
final_lr = logistic_scaled().fit(lr_train['v6'][cols_v6], y)   # refit on all 10,000 training battles

predictions_lr = final_lr.predict(lr_test['v6'][cols_v6])
submission_lr = write_submission(predictions_lr, test_ids, data.sample_submission, 'submission_lr.csv')
display(submission_lr.head())

## 5. Gradient boosting: choosing the setting

Trees can use thresholds and interactions that a linear model cannot, so gradient boosting gets the 159 features
of version 7 (version 6 plus the immunity features). `HistGradientBoostingClassifier` (`l2_regularization=1.0`,
seed 42) is tuned by a small grid on the same folds: learning rate 0.03 or 0.06, 7 or 15 leaves per tree,
300 or 600 iterations.

In [ ]:
cols_v7 = feature_columns(lr_train['v7'])
gb_search = gradient_boosting_search(cv).fit(lr_train['v7'][cols_v7], y)

grid = pd.DataFrame(gb_search.cv_results_)[
    ['param_learning_rate', 'param_max_leaf_nodes', 'param_max_iter', 'mean_test_score', 'std_test_score']
].sort_values('mean_test_score', ascending=False)
grid.columns = ['learning_rate', 'max_leaf_nodes', 'max_iter', 'CV accuracy', 'std across folds']
display(grid.style.format({'learning_rate': '{:.2f}', 'CV accuracy': '{:.2%}', 'std across folds': '{:.2%}'})
        .hide(axis='index'))

scores_gb = grid_fold_scores(gb_search)
print('Best setting:', gb_search.best_params_)

### Submission 2 · Gradient boosting (version 7)

The best setting of the grid (learning rate 0.03, 15 leaves, 300 iterations) refitted by `GridSearchCV` on all
10,000 labelled battles. Cross-validated accuracy 67.62%, within the fold-to-fold noise of the logistic regression.
On the leaderboard: 68.00% public, 67.26% private (our best private score).

In [ ]:
final_gb = gb_search.best_estimator_   # already refitted on all 10,000 training battles

predictions_gb = final_gb.predict(lr_test['v7'][cols_v7])
submission_gb = write_submission(predictions_gb, test_ids, data.sample_submission, 'submission_gb.csv')
display(submission_gb.head())

## 6. Random forest: features and validation

The random forest has its own features, built in two versions:

* **version 1 (47 features):** state of the battle at turn 30 (team HP, losses, active HP) plus how the battle
  was played: HP and loss differences at turns 10 and 20, damage dealt and recovered, switches, status and priority
  moves, boosts gained, turns spent asleep, paralysed, behind a substitute, seeded or confused;
* **version 2 (296 features):** A. team condition at turn 30 (sorted HP of the six Pokémon, HP bands, usable HP,
  HP weighted by base stats, statuses); B. type effectiveness of the attacks; C. 198 sparse columns with the
  uses of the 58 frequent moves and the HP of the 41 frequent species.

The cell below recomputes each step on the same folds, including a control grid around the default forest.

In [ ]:
rf_train, groups = build_forest_features(data.train, lookups)
rf_test, _ = build_forest_features(data.test, lookups)
print({name: len(cols) for name, cols in groups.items()})

steps = {
    'forest · state at turn 30': cv_accuracy(random_forest(), rf_train[groups['state']], y, cv),
    'forest · version 1': cv_accuracy(random_forest(), rf_train[groups['v1']], y, cv),
    'forest · + groups A, B': cv_accuracy(random_forest(), rf_train[groups['dense']], y, cv),
    'forest · + group C (version 2)': cv_accuracy(random_forest(), rf_train[groups['v2']], y, cv),
    'logistic (C=0.1) · version 2': cv_accuracy(logistic_sparse(), rf_train[groups['v2']], y, cv),
}
display(pd.DataFrame({'CV accuracy': {k: v.mean() for k, v in steps.items()},
                      'std across folds': {k: v.std() for k, v in steps.items()}})
        .style.format('{:.2%}'))

forest_search = random_forest_search(cv).fit(rf_train[groups['v1']], y)
forest_grid = pd.DataFrame(forest_search.cv_results_)[
    ['param_max_features', 'param_min_samples_leaf', 'mean_train_score', 'mean_test_score']
].sort_values('mean_test_score', ascending=False)
forest_grid.columns = ['max_features', 'min_samples_leaf', 'training accuracy', 'CV accuracy']
display(forest_grid.style.format({'training accuracy': '{:.2%}', 'CV accuracy': '{:.2%}'}).hide(axis='index'))
spread = forest_grid['CV accuracy'].max() - forest_grid['CV accuracy'].min()
print(f'Control grid on version 1: all settings within {100 * spread:.2f} points, so the defaults are kept.')

### Submission 3 · Stacked random forest (version 2)

Trees split one column at a time, so the sparse columns help the forest less than a logistic regression on the same
296 features. We therefore **stack** the two: a `StackingClassifier` gives the forest (300 trees,
`max_features=0.2`, `min_samples_leaf=5`, seed 42) the out-of-fold probability of the logistic regression as an
extra feature, together with all 296 features (`passthrough=True`). The logistic score of each training battle
comes from an inner 5-fold split, so the forest never sees a score computed on the battle itself.
Cross-validated accuracy 67.78%; the model is then refitted on all 10,000 labelled battles.
On the leaderboard: 68.67% public, 66.66% private.

In [ ]:
scores_rf = cv_accuracy(stacked_forest(), rf_train[groups['v2']], y, cv)
print(f'Stacked random forest: {scores_rf.mean():.2%} (std across folds {scores_rf.std():.2%})')

final_rf = stacked_forest().fit(rf_train[groups['v2']], y)   # refit on all 10,000 training battles

predictions_rf = final_rf.predict(rf_test[groups['v2']])
submission_rf = write_submission(predictions_rf, test_ids, data.sample_submission, 'submission_rf.csv')
display(submission_rf.head())

## 7. Summary

Fold-by-fold accuracy of the three final models on the same folds, and how often their test predictions agree.

In [ ]:
finals = {'logistic regression (v6)': scores_lr,
          'gradient boosting (v7)': scores_gb,
          'stacked random forest (v2)': scores_rf}
display(fold_table(finals, reference='logistic regression (v6)'))

predictions = {'logistic regression': predictions_lr, 'gradient boosting': predictions_gb,
               'stacked random forest': predictions_rf}
agreement = pd.DataFrame({a: {b: np.mean(predictions[a] == predictions[b]) for b in predictions}
                          for a in predictions})
print('Share of the 5,000 test battles on which two models make the same prediction:')
display(agreement.style.format('{:.1%}'))

The three models are within 0.25 points of each other in cross-validation, less than the variation between folds
(about one point), so we submitted the best-validated model of each family. All of them plateau near 68%:
with only the first 30 turns observed, the information in the timeline, more than the model, limits the accuracy.

| Submission | CV | Public | Private |
|---|---|---|---|
| Logistic regression (v6) | 67.86% | 68.67% | 66.97% |
| Gradient boosting (v7) | 67.62% | 68.00% | 67.26% |
| Stacked random forest (v2) | 67.78% | 68.67% | 66.66% |

Every submission we made, including the untuned baseline, scored higher on the public split (1,500 battles)
than on the private one (3,500 battles); our report discusses this gap.